# 🎙️ **One-Hard Voting - Baseline**

This notebook is part of the article **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

It implements the **One-Hard Voting baseline** used to evaluate the performance of DCMC-Fusion. The baseline follows a conservative decision rule: an audio sample is classified as **synthetic (fake)** if at least one of the selected deep learning models assigns a synthetic probability greater than 50%. An audio sample is classified as **natural** only when none of the selected models exceeds this threshold.

The baseline is evaluated using the same data-centric experimental design as DCMC-Fusion, considering all architecture combinations for **K = 1, 2, 3, 4, and 5**, resulting in **31 configurations**. This enables a direct comparison between One-Hard Voting and the proposed fusion approach under the same model compositions and test samples.

## Experiment: K = 1 to 5

- **Input dataset:** contains the inference probabilities generated by **20 trained models**, corresponding to five architectures—ResNet50, DenseNet121, InceptionV3, ResNet34, and VGG16—with four spectral representations per architecture: **Mel, LOG, DWT, and CQT**.

- **Ensemble dataset construction:** architecture combinations are generated for **K = 1, 2, 3, 4, and 5**. Since each architecture is represented by four trained models, a combination with **K architectures contains 4K probability columns**. This results in **5, 10, 10, 5, and 1 combinations**, respectively, for a total of **31 experimental configurations**.

- **Rows and probability values:** each row corresponds to an evaluated audio sample. Probability values range from **0 to 100**, where values closer to 0 indicate natural audio and values closer to 100 indicate synthetic audio.

- **Ground truth:** the **`label`** column identifies the true class of each audio sample: **0 = natural** and **1 = synthetic**. It is used only to evaluate the predictions and is not included in the One-Hard Voting decision.

- **One-Hard Voting:** for each architecture combination, an audio sample is classified as **synthetic** if at least one of its **4K probability values is greater than 50**. Otherwise, the sample is classified as **natural**. Therefore, a natural decision requires that all models included in the combination assign a synthetic probability less than or equal to 50.

- **Evaluation:** One-Hard Voting is applied to all **31 architecture combinations**, using the same test samples employed in the DCMC-Fusion experiments to enable a direct comparison between both fusion strategies.

# **1.  Imports**

In [2]:
from itertools import combinations
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

THRESHOLD = 50.0

ARCHITECTURES = {
    "resnet50": [
        "resnet50_mel", "resnet50_log",
        "resnet50_dwt", "resnet50_cqt"
    ],
    "densenet121": [
        "densenet121_mel", "densenet121_log",
        "densenet121_dwt", "densenet121_cqt"
    ],
    "inception_v3": [
        "inception_v3_mel", "inception_v3_log",
        "inception_v3_dwt", "inception_v3_cqt"
    ],
    "resnet34": [
        "resnet34_mel", "resnet34_log",
        "resnet34_dwt", "resnet34_cqt"
    ],
    "vgg16": [
        "vgg16_mel", "vgg16_log",
        "vgg16_dwt", "vgg16_cqt"
    ]
}

ARCH_ORDER = list(ARCHITECTURES)

FEATURE_COLUMNS = [
    column
    for columns in ARCHITECTURES.values()
    for column in columns
]

CSV_PATH = Path("inference_probabilities.csv")
SPLIT_PATH = Path("experiment_split.json")
OUTPUT_DIR = Path("results") / "baseline"

# **2.  Load and Validate Inference Probabilities**

In [3]:
df = pd.read_csv(CSV_PATH)

required_columns = ["audio", "label"] + FEATURE_COLUMNS
missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

values = df[FEATURE_COLUMNS].to_numpy(dtype=float)

if not np.isfinite(values).all() or ((values < 0) | (values > 100)).any():
    raise ValueError("Probabilities must be finite and in [0, 100].")

if set(df["label"].unique()) != {0, 1}:
    raise ValueError("Labels must contain 0 (natural) and 1 (synthetic).")

y = df["label"].to_numpy(dtype=int)

print(f"Audio samples: {len(df)}")
display(
    df["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("samples")
)


Audio samples: 2484


,samples
label,
0,1242
1,1242


# **3. Load the experimental split**

In [4]:
with open(SPLIT_PATH, encoding="utf-8") as file:
    split_configuration = json.load(file)

# Verify dataset identity before applying the saved row indices
if df["audio"].astype(str).tolist() != split_configuration["audio_order"]:
    raise ValueError("Audio samples or their order differ from the saved dataset.")

if y.tolist() != split_configuration["label_order"]:
    raise ValueError("Ground-truth labels differ from the saved dataset.")

if FEATURE_COLUMNS != split_configuration["feature_columns"]:
    raise ValueError("Feature columns differ from the search configuration.")

test_indices = np.array(
    split_configuration["test_indices"], dtype=int
)

if len(test_indices) == 0:
    raise ValueError("The saved test split is empty.")

if np.any(test_indices < 0) or np.any(test_indices >= len(df)):
    raise ValueError("The saved test indices are outside the dataset range.")

y_test = y[test_indices]

print(f"Test samples: {len(test_indices)}")
display(
    pd.Series(y_test)
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("test_samples")
)


Test samples: 746


,test_samples
label,
0,373
1,373


# **4. One-Hard Voting Rule**

For a selected architecture combination, all corresponding MEL, LOG, DWT, and CQT probabilities are considered.

For each test audio:

$$
\hat{y}_{OH} =
\begin{cases}
1, & \text{if } \displaystyle\max_{m=1,\ldots,4K} p_m > 50 \\
0, & \text{otherwise}
\end{cases}
$$

where $p_m$ represents the synthetic probability generated by the $m$-th model included in the architecture combination.

Thus, a sample is classified as **natural only when none of the selected models exceeds the 50% synthetic-probability threshold**.

In [5]:
def one_hard_predict(probabilities, threshold=THRESHOLD):
    """Return 1 (synthetic) if any selected probability is > threshold."""
    return (np.max(probabilities, axis=1) > threshold).astype(int)

# **5. Evaluate all 31 datasets**

In [6]:
global_rows = []
class_rows = []

for K in range(1, 6):
    configurations = list(combinations(ARCH_ORDER, K))

    for selected in configurations:
        configuration_name = " + ".join(selected)

        columns = [
            column
            for architecture in selected
            for column in ARCHITECTURES[architecture]
        ]

        X_test = df.iloc[test_indices][columns].to_numpy(dtype=float)
        y_pred = one_hard_predict(X_test)

        report = classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=["natural", "synthetic"],
            output_dict=True,
            zero_division=0
        )

        macro = report["macro avg"]

        base = {
            "K": K,
            "architectures": configuration_name,
            "classifier": "One-Hard"
        }

        global_rows.append({
            **base,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": macro["precision"],
            "recall_macro": macro["recall"],
            "f1_macro": macro["f1-score"]
        })

        for class_name in ["natural", "synthetic"]:
            metrics = report[class_name]

            class_rows.append({
                **base,
                "class": class_name,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1-score"],
                "support": int(metrics["support"])
            })

global_metrics = pd.DataFrame(global_rows)
class_metrics = pd.DataFrame(class_rows)

if len(global_metrics) != 31:
    raise RuntimeError(
        f"Expected 31 architecture combinations, obtained {len(global_metrics)}."
    )

print(f"Completed: {len(global_metrics)} One-Hard baseline experiments.")


Completed: 31 One-Hard baseline experiments.


# **6.  Results**

In [7]:
print("GLOBAL TEST METRICS")
display(global_metrics)

print("PER-CLASS TEST METRICS")
display(class_metrics)

GLOBAL TEST METRICS


,K,architectures,classifier,accuracy,precision_macro,recall_macro,f1_macro
0,1,resnet50,One-Hard,0.644772,0.665142,0.644772,0.633470
1,1,densenet121,One-Hard,0.632708,0.640159,0.632708,0.627761
2,1,inception_v3,One-Hard,0.595174,0.613535,0.595174,0.578118
3,1,resnet34,One-Hard,0.593834,0.607502,0.593834,0.580500
4,1,vgg16,One-Hard,0.634048,0.744982,0.634048,0.587332
5,2,resnet50 + densenet121,One-Hard,0.635389,0.671952,0.635389,0.614918
6,2,resnet50 + inception_v3,One-Hard,0.607239,0.665249,0.607239,0.569453
7,2,resnet50 + resnet34,One-Hard,0.654155,0.685833,0.654155,0.638761
8,2,resnet50 + vgg16,One-Hard,0.632708,0.778783,0.632708,0.577342
9,2,densenet121 + inception_v3,One-Hard,0.604558,0.651091,0.604558,0.571571


PER-CLASS TEST METRICS


,K,architectures,classifier,class,precision,recall,f1,support
0,1,resnet50,One-Hard,natural,0.723140,0.469169,0.569106,373
1,1,resnet50,One-Hard,synthetic,0.607143,0.820375,0.697834,373
2,1,densenet121,One-Hard,natural,0.672474,0.517426,0.584848,373
3,1,densenet121,One-Hard,synthetic,0.607843,0.747989,0.670673,373
4,1,inception_v3,One-Hard,natural,0.659193,0.394102,0.493289,373
...,...,...,...,...,...,...,...,...
57,4,resnet50 + inception_v3 + resnet34 + vgg16,One-Hard,synthetic,0.559578,0.994638,0.716216,373
58,4,densenet121 + inception_v3 + resnet34 + vgg16,One-Hard,natural,0.965116,0.222520,0.361656,373
59,4,densenet121 + inception_v3 + resnet34 + vgg16,One-Hard,synthetic,0.560606,0.991957,0.716360,373
60,5,resnet50 + densenet121 + inception_v3 + resnet...,One-Hard,natural,0.975610,0.214477,0.351648,373


**Number of experiments by K**

In [8]:
display(
    global_metrics.groupby("K")
    .size()
    .rename("experiments")
    .to_frame()
)


,experiments
K,
1,5
2,10
3,10
4,5
5,1


# **7. Save results**

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

global_path = OUTPUT_DIR / "global_metrics_one_hard.csv"
class_path = OUTPUT_DIR / "class_metrics_one_hard.csv"

global_metrics.to_csv(global_path, index=False)
class_metrics.to_csv(class_path, index=False)

print(f"Global metrics saved to: {global_path.resolve()}")
print(f"Per-class metrics saved to: {class_path.resolve()}")

Global metrics saved to: /content/results/baseline/global_metrics_one_hard.csv
Per-class metrics saved to: /content/results/baseline/class_metrics_one_hard.csv


**Output files**

The notebook generates:

- `results/baseline/global_metrics_one_hard.csv`: one row for each of the 31 architecture combinations, with accuracy and macro-averaged precision, recall, and F1-score.
- `results/baseline/class_metrics_one_hard.csv`: class-specific precision, recall, F1-score, and support for the natural and synthetic classes.

These results can be directly compared with the corresponding DCMC-Fusion configurations because the same architecture combinations and test samples are used.